# 0. Extracción de PDFs -> `icaa_raw`

Único trabajo de este notebook: leer los 5 PDFs de taquilla y espectadores y
construir `icaa_raw` (una fila por película y año en cartelera). No resuelve
`icaa_id` ni toca el catálogo ICAA -- eso es el notebook 1.

### Librerías

In [9]:
import re
import csv
from pathlib import Path

import pdfplumber
import pandas as pd

BASE = Path("..")
print("✓ Imports OK")


✓ Imports OK


In [10]:
PDFS = {
    2021: BASE / "0 - fuentes ICAA" / "taquilla_y_espectadores_2021.pdf",
    2022: BASE / "0 - fuentes ICAA" / "taquilla_y_espectadores_2022.pdf",
    2023: BASE / "0 - fuentes ICAA" / "taquilla_y_espectadores_2023.pdf",
    2024: BASE / "0 - fuentes ICAA" / "taquilla_y_espectadores_2024.pdf",
    2025: BASE / "0 - fuentes ICAA" / "taquilla_y_espectadores_2025.pdf",
}


### Distribuidoras (heurística, solo para separar el título)

Esta lista ya NO fija la distribuidora definitiva de la película -- eso lo hace
la ficha ICAA en el notebook 1. Aquí solo sirve para separar razonablemente el
título de la cola de distribuidora del PDF, para poder buscarlo luego en el
catálogo. Se guarda como `distribuidora_pdf_heuristica` (respaldo, no verdad
definitiva).

In [11]:
DISTRIBUIDORAS = sorted([
    "A Contracorriente", "Walt Disney", "Warner Bros", "Sony",
    "Universal", "Paramount Int'l", "Filmax", "Bteam Pictures",
    "DeAPlaneta", "Avalon Distribución", "Elastica", "Wanda",
    "Begin Again Films", "Syldavia", "Atalante", "Barton",
    "Atera Films", "El Sur Films", "Elamedia", "Filmin",
    "Entertainment One Films", "European Dreams Factory",
    "Phoenix Entertaiment", "Independent", "Alfa Pictures",
    "Caramel Films", "Karma", "Sherlock", "Vertice", "Summer Films",
    "Bosco Films", "Mosaico", "Benece", "Beta Fiction", "TriPictures",
    "Tri Pictures", "Version Digital", "Flins y Piniculas",
    "Splendor Films", "Surtsey Films", "Margenes Distribucion",
    "Morena Films", "Nanouk Films", "Yedra", "Hemisphere Films",
    "Stand by me Films", "Rizoma", "Baños Films", "Sideral",
    "Maravillas Distribuciones", "Acaju", "Me Lo Creo",
    "Notorious Pictures Spain", "Vercine", "Diamond Films",
    "AF Pictures", "SelectaVisión", "Nostromo", "LAZONA",
    "La Vida", "Ringo Media", "Zinea Sortzen", "Muak Canarias",
    "Urresti Producciones", "ConUnPack", "Aventura Cine",
    "Proyecfilm", "Vitrine Filmes", "Tortilla Films", "Adso Films",
    "Abre Tu Mente Distribución", "Premium", "Planeta Med",
    "Redwood Films", "Puntal Films", "Flamingo Films",
    "Caracter Films", "Acariño Films", "Minimal Films",
    "La Dalia Films", "Oliete Films", "Nekkenti",
    "Moon Entertainment", "Paycom Multimedia", "Boogaloo",
    "Selected Films", "Tiempos Dificiles Films", "Digital 104",
    "DocsBarcelona", "3boxmedia", "Jur Jur Productions",
    "Contubernio", "Cinmawings", "Las Hormigas negras",
    "Llanero Films", "Noucinemart", "Fromzero Cinema",
    "CineAND Distribución", "Ambra Projectes Culturals",
    "Potenza Producciones", "Buenavida", "Tus Ojos", "Ultreia",
    "Imagine! Factory Films", "Cosmo Fan Comunicacion",
    "Deja vu Films", "Alta Films", "Quality Media",
    "Polar Star Films", "Super 8", "Infinito mas uno",
    "Protos Films", "Hiru Damatxo", "Abanico Vision", "Film Buro",
    "Yelmo", "Avisual Concept", "Heroes AIE", "Marila Films",
    "Segarra", "Via Lactea", "Arena Comunicacion", "Wanda - Avalon",
    "El Deseo", "Sheridan's Producciones", "Acfilms", "Inflamavle",
    "Estudi Playtime", "Malvalanda", "El Dedo en el Ojo", "Empatia",
    "Zeta", "Paco Poch Cinema", "Orio", "Ollo Vivo",
    "Oscar Parada Castellano", "Alberto Jose Redondo Villa",
    "OmniCorp Estudio", "Disentropic", "The Social Dog",
    "The Other Side Films", "Pleamar Films", "Teaser Films",
    "Dada Films", "Jambika Docs", "Pecker", "Raabta Pictures",
    "Vendaval", "Franjiverde", "Sin Parpadear", "Silencio Cinema",
    "Gondola Films", "Jaibo Films", "Gat-alana", "Cameo",
    "Taranna Films", "Les Films de la Resistance",
    "Boccacio Distribucion", "Mercury Films", "Reverso Films",
    "Enciende Television", "Mubox.Studio", "Zeitun Films",
    "RTVE", "Aquelarre", "La Luna de Tantan", "Txintxua Films",
    "Eigakan Films", "Cuerda Floja", "Plan Secreto",
    "No Tan Chalados", "TCM", "Lost&Found", "Piramide Films",
    "Quechua Films", "Sophia Network", "Festival Films",
    "39 Escalones", "Golem", "El Ojo Mecanico", "Artistic Metropol",
    "Rita & Luca", "Noon Films", "New Frecuency", "Wild Stories",
    "Lasdelcine Producciones Aud.", "Barlovento Distribucion",
    "Vertigo Films", "Magenta Films Distribución", "Aguilar Cinema",
    "La Esgueva Films", "Maleta Films", "Molonko Films",
    "Marvin & Wayne", "Jose Texeira", "Son Pelis",
    "Films 59", "Cinebinario",
], key=len, reverse=True)


PATRON = re.compile(
    r'^(\d+)\s+'
    r'(.+?)\s+'
    r'(\d{2}/\d{2}/\d{4})\s+'
    r'([\d.,]+)\s*€?\s+'
    r'([\d.,]+)$'
)

def separar_titulo_distribuidora(titulo_raw):
    titulo_raw = titulo_raw.strip()
    for dist in DISTRIBUIDORAS:
        if titulo_raw.endswith(dist):
            titulo = titulo_raw[:-len(dist)].strip()
            return titulo, dist
        if f' {dist}' in titulo_raw:
            idx = titulo_raw.rfind(f' {dist}')
            titulo = titulo_raw[:idx].strip()
            return titulo, dist
    return titulo_raw, None

def extraer_pdf(pdf_path, anio):
    rows = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            texto = page.extract_text()
            if not texto:
                continue
            for linea in texto.split('\n'):
                linea = linea.strip()
                m = PATRON.match(linea)
                if m:
                    rows.append({
                        'titulo_raw': m.group(2).strip(),
                        'fecha_estreno': m.group(3),
                        'recaudacion': float(m.group(4).replace('.', '').replace(',', '.')),
                        'espectadores': int(m.group(5).replace('.', '')),
                        'anio': anio,
                    })
    df = pd.DataFrame(rows)
    df[['titulo', 'distribuidora_pdf_heuristica']] = df['titulo_raw'].apply(
        lambda x: pd.Series(separar_titulo_distribuidora(x))
    )
    return df[['titulo', 'distribuidora_pdf_heuristica', 'fecha_estreno',
               'recaudacion', 'espectadores', 'anio']]

print("✓ funciones de extracción de PDF definidas")


✓ funciones de extracción de PDF definidas


In [12]:
icaa_raw = pd.concat(
    [extraer_pdf(ruta, anio) for anio, ruta in PDFS.items()],
    ignore_index=True,
)
print(f"icaa_raw: {len(icaa_raw)} filas")
icaa_raw.head()


icaa_raw: 2812 filas


,titulo,distribuidora_pdf_heuristica,fecha_estreno,recaudacion,espectadores,anio
0,A todo tren. Destino Asturias,Warner Bros,08/07/2021,8493358.0,1500811,2021
1,Way Down,Sony,12/11/2021,5628247.0,887897,2021
2,Operacion Camaron,Walt Disney,24/06/2021,3522415.0,597700,2021
3,"Buen patron, El",Tri Pictures,15/10/2021,3336892.0,528523,2021
4,Maixabel,Walt Disney,24/09/2021,2828416.0,515293,2021


### Limpiar sufijos de reposición y separar el artículo pospuesto

Muchos títulos del PDF llevan anotaciones de reposición que nunca están en el
título real del catálogo ICAA: `(1961) (re)`, `(re)`, `(4k re)`, `(re:2021)`,
`(re: 2016)`, `25 Aniversario`... Si se buscan tal cual, la búsqueda no
encuentra nada (ni por subcadena), porque esa anotación no forma parte del
título oficial. Se limpia esto **antes** de separar el artículo (el sufijo va
después de la coma+artículo, así que hay que quitarlo primero para que la
detección del artículo funcione).

Convención igual que antes:
- `titulo` -- el original, sin tocar (clave de caché/overrides, no cambia)
- `articulo` -- el artículo detectado, o `None`
- `titulo_busqueda` -- cuerpo limpio de reposición y sin artículo, para buscar
  por subcadena en el catálogo

In [13]:
import re as _re  # ya importado arriba, alias por claridad local

def limpiar_reposicion(titulo):
    """
    Quita anotaciones de reposición/año que no forman parte del título real
    del catálogo ICAA: (1961), (re), (4k re), (re:2021), (re: 2016), (Re),
    y sufijos de aniversario ("... 25 Aniversario"). Además CAPTURA el año
    de producción cuando aparece como "(YYYY)" -- ese año suele ser el año
    de producción real de la película, muy útil para desambiguar reposiciones
    donde fecha_estreno es la fecha de la REPOSICIÓN, no del estreno original
    (pueden diferir varias décadas).
    Devuelve: (titulo_limpio, anio_produccion_o_None)
    """
    t = titulo.strip()
    anio_produccion = None

    m_anio = _re.search(
        r"\((\d{4})\)\s*(?:\((?:4k\s*)?re\s*:?\s*\d{0,4}\s*\)\s*)?(?:\s*\d{1,3}\s*[AaÁá]niversario)?\s*$",
        t, _re.IGNORECASE,
    )
    if m_anio:
        anio_produccion = int(m_anio.group(1))

    t = _re.sub(r"\s*\d{1,3}\s*[AaÁá]niversario\s*$", "", t)
    patron = _re.compile(
        r"\s*\(\s*(?:\d{4}\s*)?(?:4k\s*)?(?:re\s*:?\s*\d{0,4}\s*)?\)\s*$",
        _re.IGNORECASE,
    )
    cambiado = True
    while cambiado:
        nuevo_t = patron.sub("", t)
        cambiado = (nuevo_t != t)
        t = nuevo_t
    return t.strip(), anio_produccion


def separar_articulo(titulo):
    """
    Detecta el patrón 'Cuerpo, Artículo' al final del título. Devuelve
    (cuerpo_sin_articulo, articulo) o (titulo, None) si no aplica.
    """
    m = _re.match(r"^(.+),\s*(El|La|Los|Las)$", titulo.strip(), flags=_re.IGNORECASE)
    if m:
        cuerpo = m.group(1).strip()
        articulo = m.group(2).strip()
        return cuerpo, articulo
    return titulo.strip(), None


limpieza = icaa_raw["titulo"].apply(limpiar_reposicion)
icaa_raw["titulo_limpio_reposicion"] = limpieza.apply(lambda x: x[0])
icaa_raw["anio_reposicion"] = limpieza.apply(lambda x: x[1])
icaa_raw[["titulo_busqueda", "articulo"]] = icaa_raw["titulo_limpio_reposicion"].apply(
    lambda t: pd.Series(separar_articulo(t))
)
icaa_raw = icaa_raw.drop(columns=["titulo_limpio_reposicion"])

con_articulo = icaa_raw["articulo"].notna().sum()
con_anio_reposicion = icaa_raw["anio_reposicion"].notna().sum()
print(f"Títulos con artículo pospuesto detectado: {con_articulo} / {len(icaa_raw)}")
print(f"Títulos con año de producción capturado de una reposición: {con_anio_reposicion} / {len(icaa_raw)}")
icaa_raw[icaa_raw["anio_reposicion"].notna()][
    ["titulo", "titulo_busqueda", "articulo", "anio_reposicion"]
].head(10)


Títulos con artículo pospuesto detectado: 659 / 2812
Títulos con año de producción capturado de una reposición: 208 / 2812


,titulo,titulo_busqueda,articulo,anio_reposicion
94,"Dia de la bestia, El (1995) (4k re)",Dia de la bestia,El,1995.0
101,"Escopeta nacional, La (1978)",Escopeta nacional,La,1978.0
178,"Extraño viaje, El (1964)",Extraño viaje,El,1964.0
216,Moros y Cristianos (1987),Moros y Cristianos,NaN,1987.0
225,"Jueves, milagro, Los (1957) (re)","Jueves, milagro",Los,1957.0
229,Placido (1961) (re),Placido,NaN,1961.0
233,Calabuch (1956) (re),Calabuch,NaN,1956.0
236,"Reportero, El (1975) (re)",Reportero,El,1975.0
238,"Vida por delante, La (1958) (re)",Vida por delante,La,1958.0
241,Esa pareja feliz (1951) (re),Esa pareja feliz,NaN,1951.0


### Guardar para el notebook 1

In [14]:
RUTA_ICAA_RAW = BASE / "3 - csv" / "icaa_raw_pdfs.csv"
icaa_raw.to_csv(RUTA_ICAA_RAW, index=False, sep=';', quoting=csv.QUOTE_NONNUMERIC)
print(f"✓ Guardado -> {RUTA_ICAA_RAW}")


✓ Guardado -> ..\3 - csv\icaa_raw_pdfs.csv
